# Bounded NONAN healthy-enrichment ablation

Compare the unchanged three-source model with a source-aware, capped-NONAN variant. All splits are by participant; frozen NONAN and RevalExo are not read.

In [1]:
from pathlib import Path
import sys
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
ROOT = Path.cwd().resolve(); ROOT = ROOT.parent if ROOT.name.lower() == 'notebooks' else ROOT
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from models.stroke_gait_inception import StrokeGaitInception
P = ROOT / 'data' / 'processed'; N = ROOT / 'data' / 'interim' / 'nonan_gaitprint'
D = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:', D)
torch.manual_seed(42); np.random.seed(42)
x = np.concatenate([np.load(P / 'validated_acceleration_magnitude_windows_float32.npy'), np.load(P / 'sint_maartenskliniek_external_windows_float32.npy')])
m = pd.concat([pd.read_csv(P / 'validated_window_metadata.csv'), pd.read_csv(P / 'sint_maartenskliniek_external_window_metadata.csv')], ignore_index=True)
m = m.loc[m.label.isin(['healthy', 'stroke'])].reset_index(drop=True); m['y'] = m.label.eq('stroke').astype(int); m['source'] = m.dataset_id; m['group'] = m.participant_key.astype(str)
nx = np.load(N / 'candidate_healthy_enrichment_magnitude_isolated_spike_repaired.npy', mmap_mode='r')
nm = pd.read_csv(N / 'candidate_healthy_enrichment_window_metadata.csv'); nm['y'] = 0; nm['source'] = 'nonan_gaitprint'; nm['group'] = nm.participant_key.astype(str)
# Fixed 64-window cap per NONAN participant prevents window-count dominance.
rng = np.random.default_rng(42); keep = np.concatenate([rng.choice(v, min(64, len(v)), replace=False) for v in nm.groupby('group').indices.values()])
nx = np.asarray(nx[keep]); nm = nm.iloc[keep].reset_index(drop=True)
people = pd.concat([m[['group', 'source', 'y']].drop_duplicates(), nm[['group', 'source', 'y']].drop_duplicates()], ignore_index=True)
people['stratum'] = people.source + '|' + people.y.astype(str); folds = StratifiedKFold(3, shuffle=True, random_state=42)
rows = []
for fold, (trp, vap) in enumerate(folds.split(people, people.stratum)):
    trgroups, vagroups = set(people.iloc[trp].group), set(people.iloc[vap].group)
    bt, bv = m.group.isin(trgroups).to_numpy(), m.group.isin(vagroups).to_numpy()
    nt, nv = nm.group.isin(trgroups).to_numpy(), nm.group.isin(vagroups).to_numpy()
    for mode in ['baseline', 'nonan_capped_10pct']:
        tx, tm = x[bt], m.loc[bt].copy()
        if mode != 'baseline': tx, tm = np.concatenate([tx, nx[nt]]), pd.concat([tm, nm.loc[nt]], ignore_index=True)
        mean, std = tx.reshape(-1, 3).mean(0), tx.reshape(-1, 3).std(0).clip(1e-4)
        z = torch.from_numpy(((tx - mean) / std).transpose(0, 2, 1).astype('float32')); y = torch.from_numpy(tm.y.to_numpy('float32'))
        keys = tm.source + '|' + tm.y.astype(str); counts = keys.value_counts(); cap = tm.source.map(lambda s: .1 if s == 'nonan_gaitprint' else 1.0)
        w = torch.tensor((cap / keys.map(counts)).to_numpy(), dtype=torch.double)
        dl = DataLoader(TensorDataset(z, y), 128, sampler=WeightedRandomSampler(w, len(w), replacement=True))
        net = StrokeGaitInception().to(D); opt = torch.optim.AdamW(net.parameters(), 1e-3, weight_decay=1e-4)
        for _ in range(6):
            net.train()
            for a, b in dl:
                opt.zero_grad(); loss = torch.nn.functional.binary_cross_entropy_with_logits(net(a.to(D)), b.to(D)); loss.backward(); opt.step()
        net.eval()
        for name, ex, em in [('original', x[bv], m.loc[bv]), ('nonan_holdout', nx[nv], nm.loc[nv])]:
            with torch.inference_mode(): p = torch.sigmoid(net(torch.from_numpy(((ex - mean) / std).transpose(0, 2, 1).astype('float32')).to(D))).cpu().numpy()
            g = em.assign(p=p).groupby(['group', 'y'], as_index=False).p.mean()
            row = {'fold': fold, 'mode': mode, 'evaluation': name, 'participants': len(g), 'healthy': int((g.y == 0).sum()), 'stroke': int((g.y == 1).sum()), 'balanced_accuracy': balanced_accuracy_score(g.y, g.p >= .5) if g.y.nunique() == 2 else np.nan, 'healthy_specificity': float((g.loc[g.y == 0, 'p'] < .5).mean()), 'auroc': roc_auc_score(g.y, g.p) if g.y.nunique() == 2 else np.nan}
            rows.append(row); print(row)
out = pd.DataFrame(rows); out.to_csv(P / 'bounded_nonan_enrichment_ablation.csv', index=False)
print(out.groupby(['mode', 'evaluation'])[['auroc', 'balanced_accuracy', 'healthy_specificity']].mean())

device: cuda


{'fold': 0, 'mode': 'baseline', 'evaluation': 'original', 'participants': 105, 'healthy': 42, 'stroke': 63, 'balanced_accuracy': 0.7698412698412698, 'healthy_specificity': 0.9523809523809523, 'auroc': 0.9402872260015116}
{'fold': 0, 'mode': 'baseline', 'evaluation': 'nonan_holdout', 'participants': 27, 'healthy': 27, 'stroke': 0, 'balanced_accuracy': nan, 'healthy_specificity': 0.9629629629629629, 'auroc': nan}


{'fold': 0, 'mode': 'nonan_capped_10pct', 'evaluation': 'original', 'participants': 105, 'healthy': 42, 'stroke': 63, 'balanced_accuracy': 0.8531746031746033, 'healthy_specificity': 0.8333333333333334, 'auroc': 0.9482237339380195}
{'fold': 0, 'mode': 'nonan_capped_10pct', 'evaluation': 'nonan_holdout', 'participants': 27, 'healthy': 27, 'stroke': 0, 'balanced_accuracy': nan, 'healthy_specificity': 0.8888888888888888, 'auroc': nan}


{'fold': 1, 'mode': 'baseline', 'evaluation': 'original', 'participants': 105, 'healthy': 43, 'stroke': 62, 'balanced_accuracy': 0.8488372093023255, 'healthy_specificity': 0.6976744186046512, 'auroc': 0.9984996249062266}
{'fold': 1, 'mode': 'baseline', 'evaluation': 'nonan_holdout', 'participants': 26, 'healthy': 26, 'stroke': 0, 'balanced_accuracy': nan, 'healthy_specificity': 0.7692307692307693, 'auroc': nan}


{'fold': 1, 'mode': 'nonan_capped_10pct', 'evaluation': 'original', 'participants': 105, 'healthy': 43, 'stroke': 62, 'balanced_accuracy': 0.9606151537884471, 'healthy_specificity': 0.9534883720930233, 'auroc': 0.9966241560390098}
{'fold': 1, 'mode': 'nonan_capped_10pct', 'evaluation': 'nonan_holdout', 'participants': 26, 'healthy': 26, 'stroke': 0, 'balanced_accuracy': nan, 'healthy_specificity': 1.0, 'auroc': nan}


{'fold': 2, 'mode': 'baseline', 'evaluation': 'original', 'participants': 104, 'healthy': 41, 'stroke': 63, 'balanced_accuracy': 0.8284939992257065, 'healthy_specificity': 0.926829268292683, 'auroc': 0.9260549748354626}
{'fold': 2, 'mode': 'baseline', 'evaluation': 'nonan_holdout', 'participants': 27, 'healthy': 27, 'stroke': 0, 'balanced_accuracy': nan, 'healthy_specificity': 1.0, 'auroc': nan}


{'fold': 2, 'mode': 'nonan_capped_10pct', 'evaluation': 'original', 'participants': 104, 'healthy': 41, 'stroke': 63, 'balanced_accuracy': 0.8443670150987224, 'healthy_specificity': 0.926829268292683, 'auroc': 0.9183120402632597}
{'fold': 2, 'mode': 'nonan_capped_10pct', 'evaluation': 'nonan_holdout', 'participants': 27, 'healthy': 27, 'stroke': 0, 'balanced_accuracy': nan, 'healthy_specificity': 1.0, 'auroc': nan}
                                     auroc  balanced_accuracy  \
mode               evaluation                                   
baseline           nonan_holdout       NaN                NaN   
                   original       0.954947           0.815724   
nonan_capped_10pct nonan_holdout       NaN                NaN   
                   original       0.954387           0.886052   

                                  healthy_specificity  
mode               evaluation                          
baseline           nonan_holdout             0.910731  
                   ori

Accept only if original-source AUROC/balanced accuracy do not materially decline and held-out candidate healthy specificity improves or is maintained. Frozen cohorts remain unscored in this ablation.